# jurong_spatial_lag_models
Sanitized research code. Default input is SYNTHETIC, not paper data. Run via tools/run_models.py. See docs/MANUSCRIPT_ALIGNMENT.md.


In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D

from libpysal.weights import Queen, KNN
from esda.moran import Moran
from spreg import ML_Lag

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*The weights matrix is not fully connected.*')
warnings.filterwarnings('ignore', message='.*The weights matrix contains islands.*')

# Public package: never search parent folders for private data.
import os
from IPython.display import display
PROJECT_ROOT = Path(os.environ['UFUG_PUBLIC_ROOT']).resolve()
DATA_PATH = Path(os.environ.get('UFUG_MODEL_DATA', str(PROJECT_ROOT / 'data/synthetic/model_cells.geojson'))).resolve()
DATA_RELATIVE = DATA_PATH
OUTPUT_DIR = Path(os.environ.get('UFUG_OUTPUT_ROOT', str(PROJECT_ROOT / 'outputs'))) / "jurong_spatial_lag_models"
if DATA_PATH != (PROJECT_ROOT / 'data/synthetic/model_cells.geojson').resolve() and OUTPUT_DIR.resolve().is_relative_to(PROJECT_ROOT):
    raise ValueError('Private model outputs must resolve outside Public, including redirected model directories')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


KNN_FALLBACK_K = 8
MORAN_PERMUTATIONS = 999
RANDOM_SEED = 20260420

OUTCOMES = {
    'log_F_h1': {
        'source': 'F_h',
        'label': 'Visit frequency',
        'axis_label': 'log(visit frequency + 1)',
        'stem': 'visit_frequency',
    },
    'log_D_h1': {
        'source': 'D_h',
        'label': 'Visit duration',
        'axis_label': 'log(visit duration + 1)',
        'stem': 'visit_duration',
    },
}

X_COLS = ['ps', 'pl', 'pb', 'pm', 'peg', 'peo', 'pep', 'peps', 'per', 'pepw', 'set']
LABEL_MAP = {
    'ps': 'Perceptual safety',
    'pl': 'Perceptual liveliness',
    'pb': 'Perceptual beauty',
    'pm': 'Perceptual monotony',
    'peg': 'Greenness',
    'peo': 'Openness',
    'pep': 'Furniture',
    'peps': 'Proximity to park services',
    'per': 'Road connectivity',
    'pepw': 'Proximity to water',
    'set': 'Set',
}
PALETTE = {
    'Visit frequency': '#ba121f',
    'Visit duration': '#08306b',
}

mpl.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'legend.title_fontsize': 8,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.7,
    'axes.grid': False,
})
sns.set_theme(style='white', context='paper')

print('Project root:', PROJECT_ROOT)
print('Data path:', DATA_PATH.name)
print('Output directory:', OUTPUT_DIR.name)


In [ ]:

gdf = gpd.read_file(DATA_PATH)

required_cols = X_COLS + ['F_h', 'D_h', 'park', 'geometry']
missing = [col for col in required_cols if col not in gdf.columns]
if missing:
    raise KeyError(f'Missing required columns: {missing}')

for col in X_COLS + ['F_h', 'D_h']:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce')

jurong = gdf[gdf['park'].astype(str).str.lower().eq('jurong')].copy()
jurong['log_F_h1'] = np.log1p(jurong['F_h'])
jurong['log_D_h1'] = np.log1p(jurong['D_h'])

model_cols = X_COLS + list(OUTCOMES) + ['grid_id', 'F_h', 'D_h', 'park', 'geometry']
model_data = jurong.dropna(subset=X_COLS + list(OUTCOMES)).reset_index(drop=True)

summary = pd.DataFrame({
    'park': ['Jurong Lake Gardens'],
    'available_hexagons': [len(jurong)],
    'model_hexagons': [len(model_data)],
    'total_visits': [model_data['F_h'].sum()],
    'mean_visits': [model_data['F_h'].mean()],
    'total_duration': [model_data['D_h'].sum()],
    'mean_duration': [model_data['D_h'].mean()],
})
display(summary.round(2))


In [ ]:

def build_weights(sub_gdf, k=KNN_FALLBACK_K):
    queen = Queen.from_dataframe(sub_gdf, use_index=True)
    if queen.n_components == 1 and not queen.islands:
        queen.transform = 'R'
        return queen, 'Queen contiguity'

    k_eff = min(k, len(sub_gdf) - 1)
    knn = KNN.from_dataframe(sub_gdf, k=k_eff)
    knn.transform = 'R'
    return knn, f'KNN, k={k_eff}'

w, weight_label = build_weights(model_data)
print(f'Weights: {weight_label}; n={w.n}; components={w.n_components}; islands={len(w.islands)}')

moran_rows = []
for y_col, meta in OUTCOMES.items():
    np.random.seed(RANDOM_SEED)
    moran = Moran(model_data[y_col].to_numpy(), w, permutations=MORAN_PERMUTATIONS)
    moran_rows.append({
        'outcome': meta['label'],
        'outcome_variable': y_col,
        'n': w.n,
        'weights': weight_label,
        'moran_i': moran.I,
        'expected_i': moran.EI,
        'z_norm': moran.z_norm,
        'p_norm': moran.p_norm,
        'p_sim': moran.p_sim,
        'permutations': MORAN_PERMUTATIONS,
    })

moran_table = pd.DataFrame(moran_rows)
display(moran_table.round(4))


In [ ]:

analysis_data = model_data.copy()
standardization_rows = []
for col in X_COLS:
    mean = analysis_data[col].mean()
    std = analysis_data[col].std(ddof=0)
    if pd.isna(std) or np.isclose(std, 0):
        analysis_data[col] = 0.0
    else:
        analysis_data[col] = (analysis_data[col] - mean) / std
    standardization_rows.append({'term': col, 'label': LABEL_MAP[col], 'mean': mean, 'std_ddof0': std})

standardization_table = pd.DataFrame(standardization_rows)
X = analysis_data[X_COLS].to_numpy()

models = {}
for y_col, meta in OUTCOMES.items():
    y = analysis_data[[y_col]].to_numpy()
    models[meta['label']] = ML_Lag(
        y=y,
        x=X,
        w=w,
        name_y=y_col,
        name_x=X_COLS,
        name_w=weight_label,
        name_ds='Jurong Lake Gardens',
        method='full',
    )
    print(f"Fitted SLM for {meta['label']}")


In [ ]:

def stars(p):
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    if p < 0.10:
        return '+' 
    return ''

def significance_group(p):
    if p < 0.05:
        return 'p < 0.05'
    if p < 0.10:
        return '0.05 <= p < 0.10'
    return 'p >= 0.10'

def clean_term(term):
    if term == 'CONSTANT':
        return 'Intercept'
    if term.startswith('W_'):
        return 'Spatial lag (rho)'
    return LABEL_MAP.get(term, term)

def safe_to_csv(df, filename):
    output_path = OUTPUT_DIR / filename
    try:
        df.to_csv(output_path, index=False)
        return output_path
    except PermissionError:
        fallback_path = output_path.with_name(f'{output_path.stem}_corrected{output_path.suffix}')
        df.to_csv(fallback_path, index=False)
        print(f'Skipped locked file: {output_path}; wrote {fallback_path.name}')
        return fallback_path

def safe_to_geojson(gdf_to_write, filename):
    output_path = OUTPUT_DIR / filename
    try:
        gdf_to_write.to_file(output_path, driver='GeoJSON', index=False)
        return output_path
    except PermissionError:
        fallback_path = output_path.with_name(f'{output_path.stem}_corrected{output_path.suffix}')
        gdf_to_write.to_file(fallback_path, driver='GeoJSON', index=False)
        print(f'Skipped locked file: {output_path}; wrote {fallback_path.name}')
        return fallback_path

def save_figure(fig, stem):
    saved = []
    for ext in ['png', 'svg', 'pdf']:
        output_path = OUTPUT_DIR / f'{stem}.{ext}'
        try:
            fig.savefig(output_path)
            saved.append(output_path)
        except PermissionError:
            print(f'Skipped locked file: {output_path}')
    return saved

zcrit = 1.959963984540054
coef_rows = []
fit_rows = []
prediction_frames = []
residual_gdf = model_data[['grid_id', 'geometry']].copy()

for outcome_label, model in models.items():
    y_col = model.name_y
    meta = next(v for v in OUTCOMES.values() if v['label'] == outcome_label)

    for idx, term in enumerate(model.name_x):
        estimate = float(model.betas[idx, 0])
        std_error = float(model.std_err[idx])
        z_value, p_value = model.z_stat[idx]
        z_value = float(z_value)
        p_value = float(p_value)
        coef_rows.append({
            'outcome': outcome_label,
            'outcome_variable': y_col,
            'term': term,
            'label': clean_term(term),
            'estimate': estimate,
            'std_error': std_error,
            'z_value': z_value,
            'p_value': p_value,
            'ci_low': estimate - zcrit * std_error,
            'ci_high': estimate + zcrit * std_error,
            'stars': stars(p_value),
            'significance': significance_group(p_value),
        })

    rho_idx = [i for i, term in enumerate(model.name_x) if term.startswith('W_')][0]
    rho_z, rho_p = model.z_stat[rho_idx]
    fit_rows.append({
        'outcome': outcome_label,
        'outcome_variable': y_col,
        'n': int(model.n),
        'k': int(model.k),
        'weights': weight_label,
        'rho': float(model.rho),
        'rho_z': float(rho_z),
        'rho_p_value': float(rho_p),
        'log_likelihood': float(model.logll),
        'aic': float(model.aic),
        'schwarz_bic': float(model.schwarz),
        'pseudo_r2': float(model.pr2),
        'spatial_pseudo_r2': float(model.pr2_e),
        'sigma2': float(model.sig2),
    })

    pred = model_data[['grid_id', 'park', 'F_h', 'D_h', 'log_F_h1', 'log_D_h1']].copy()
    pred['outcome'] = outcome_label
    pred['outcome_variable'] = y_col
    pred['observed'] = analysis_data[y_col].to_numpy()
    pred['fitted'] = model.predy.flatten()
    pred['residual'] = model.u.flatten()
    prediction_frames.append(pred)

    stem = meta['stem']
    residual_gdf[f'{stem}_observed'] = analysis_data[y_col].to_numpy()
    residual_gdf[f'{stem}_fitted'] = model.predy.flatten()
    residual_gdf[f'{stem}_residual'] = model.u.flatten()

coef_table = pd.DataFrame(coef_rows)
fit_table = pd.DataFrame(fit_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)

safe_to_csv(coef_table, 'jurong_slm_coefficients.csv')
safe_to_csv(fit_table, 'jurong_slm_model_fit.csv')
safe_to_csv(moran_table, 'jurong_slm_moran.csv')
safe_to_csv(predictions, 'jurong_slm_predictions.csv')
safe_to_csv(standardization_table, 'jurong_slm_standardization.csv')
safe_to_geojson(residual_gdf, 'jurong_slm_predictions.geojson')

latex = coef_table.copy()
latex['estimate_se'] = latex.apply(lambda r: f"{r['estimate']:.3f}{r['stars']} ({r['std_error']:.3f})", axis=1)
latex_wide = latex.pivot(index='label', columns='outcome', values='estimate_se')
try:
    latex_wide.to_latex(OUTPUT_DIR / 'jurong_slm_coefficients.tex', escape=True)
except PermissionError:
    fallback_latex_path = OUTPUT_DIR / 'jurong_slm_coefficients_corrected.tex'
    latex_wide.to_latex(fallback_latex_path, escape=True)
    print(f'Skipped locked LaTeX file; wrote {fallback_latex_path.name}')

print('Saved outputs to', OUTPUT_DIR.name)
display(fit_table.round(4))
display(coef_table.round(4))


In [ ]:

plot_df = coef_table[coef_table['term'].isin(X_COLS)].copy()
order = (
    plot_df.groupby('label', sort=False)['estimate']
    .apply(lambda s: s.abs().max())
    .sort_values()
    .index.tolist()
)
y_positions = np.arange(len(order))
y_lookup = {label: idx for idx, label in enumerate(order)}

fig, ax = plt.subplots(figsize=(8.2, 4.8))
offset = 0.16
x_min = float(plot_df['ci_low'].min())
x_max = float(plot_df['ci_high'].max())
x_span = x_max - x_min if not np.isclose(x_max, x_min) else 1.0
x_pad = x_span * 0.025

for outcome_idx, outcome_label in enumerate(OUTCOMES[y]['label'] for y in OUTCOMES):
    sub = plot_df[plot_df['outcome'] == outcome_label].copy()
    sub['y'] = sub['label'].map(y_lookup).astype(float) + (outcome_idx - 0.5) * 2 * offset

    for sig_level in ['p >= 0.10', '0.05 <= p < 0.10', 'p < 0.05']:
        part = sub[sub['significance'] == sig_level]
        if part.empty:
            continue
        if sig_level == 'p < 0.05':
            color = PALETTE[outcome_label]
            face = color
            edge = color
            markersize = 5.8
            alpha = 0.98
            zorder = 4
            line_width = 1.2
        elif sig_level == '0.05 <= p < 0.10':
            color = PALETTE[outcome_label]
            face = 'white'
            edge = color
            markersize = 5.2
            alpha = 0.86
            zorder = 3
            line_width = 1.05
        else:
            color = '#b8b8b8'
            face = 'white'
            edge = '#777777'
            markersize = 4.4
            alpha = 0.62
            zorder = 2
            line_width = 0.9

        estimates = part['estimate'].to_numpy(dtype=float)
        y_vals = part['y'].to_numpy(dtype=float)
        ci_low = part['ci_low'].to_numpy(dtype=float)
        ci_high = part['ci_high'].to_numpy(dtype=float)
        xerr = np.vstack([estimates - ci_low, ci_high - estimates])
        ax.errorbar(
            estimates,
            y_vals,
            xerr=xerr,
            fmt='o',
            markersize=markersize,
            capsize=2.5,
            elinewidth=line_width,
            color=color,
            ecolor=color,
            markerfacecolor=face,
            markeredgecolor=edge,
            markeredgewidth=1.1,
            alpha=alpha,
            zorder=zorder,
        )

    for row in sub[sub['significance'] == 'p < 0.05'].itertuples():
        ax.text(row.ci_high + x_pad, row.y, '*', color=PALETTE[outcome_label], va='center', ha='left', fontsize=9, fontweight='bold')
    for row in sub[sub['significance'] == '0.05 <= p < 0.10'].itertuples():
        ax.text(row.ci_high + x_pad, row.y, '+', color=PALETTE[outcome_label], va='center', ha='left', fontsize=8.5)

ax.axvline(0, color='#333333', linewidth=0.8, linestyle='--')
ax.set_yticks(y_positions)
ax.set_yticklabels(order)
ax.set_xlim(x_min - x_pad * 3, x_max + x_pad * 8)
ax.set_xlabel('Standardized coefficient, 95% ML confidence interval')
ax.set_ylabel('')
ax.set_title('Spatial lag models for Jurong Lake Gardens')

outcome_handles = [
    Line2D([0], [0], marker='o', linestyle='', color=PALETTE[label], markerfacecolor=PALETTE[label], markeredgecolor=PALETTE[label], label=label)
    for label in PALETTE
]
sig_handles = [
    Line2D([0], [0], marker='o', linestyle='', color='#333333', markerfacecolor='#333333', markeredgecolor='#333333', label='p < 0.05'),
    Line2D([0], [0], marker='o', linestyle='', color='#555555', markerfacecolor='white', markeredgecolor='#555555', label='0.05 <= p < 0.10'),
    Line2D([0], [0], marker='o', linestyle='', color='#777777', markerfacecolor='white', markeredgecolor='#777777', label='p >= 0.10'),
]
legend_outcome = ax.legend(handles=outcome_handles, title='Outcome', frameon=False, loc='lower left', bbox_to_anchor=(1.01, 0.05), borderaxespad=0)
ax.add_artist(legend_outcome)
ax.legend(handles=sig_handles, title='Significance', frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1.0), borderaxespad=0)
ax.grid(axis='x', color='#dddddd', linewidth=0.7)
ax.grid(axis='y', visible=False)
sns.despine(ax=ax, left=True, bottom=False)
fig.tight_layout(rect=[0, 0, 0.82, 1])
save_figure(fig, 'jurong_slm_standardized_coefficients')
display(fig)
plt.close(fig)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.4))

for ax, (y_col, meta) in zip(axes, OUTCOMES.items()):
    outcome_label = meta['label']
    sub = predictions[predictions['outcome'] == outcome_label]
    color = PALETTE[outcome_label]
    ax.scatter(
        sub['observed'],
        sub['fitted'],
        s=22,
        color=color,
        alpha=0.72,
        edgecolor='white',
        linewidth=0.35,
    )
    low = min(float(sub['observed'].min()), float(sub['fitted'].min()))
    high = max(float(sub['observed'].max()), float(sub['fitted'].max()))
    pad = (high - low) * 0.04 if not np.isclose(high, low) else 0.5
    ax.plot([low - pad, high + pad], [low - pad, high + pad], color='#333333', linewidth=0.8, linestyle='--')
    pseudo_r2 = float(fit_table.loc[fit_table['outcome'] == outcome_label, 'pseudo_r2'].iloc[0])
    rho = float(fit_table.loc[fit_table['outcome'] == outcome_label, 'rho'].iloc[0])
    ax.text(0.04, 0.96, f'rho = {rho:.3f}\npseudo R2 = {pseudo_r2:.3f}', transform=ax.transAxes, va='top', ha='left', fontsize=8)
    ax.set_title(outcome_label)
    ax.set_xlabel(f"Observed {meta['axis_label']}")
    ax.set_ylabel(f"Fitted {meta['axis_label']}")
    ax.set_xlim(low - pad, high + pad)
    ax.set_ylim(low - pad, high + pad)
    ax.grid(color='#dddddd', linewidth=0.6)
    sns.despine(ax=ax)

fig.tight_layout()
save_figure(fig, 'jurong_slm_observed_vs_fitted')
display(fig)
plt.close(fig)


In [ ]:

resid_cols = [f"{meta['stem']}_residual" for meta in OUTCOMES.values()]
max_abs = max(float(residual_gdf[col].abs().max()) for col in resid_cols)
max_abs = max_abs if max_abs > 0 else 1.0

fig, axes = plt.subplots(1, 2, figsize=(7.8, 3.8))
for ax, (y_col, meta) in zip(axes, OUTCOMES.items()):
    resid_col = f"{meta['stem']}_residual"
    residual_gdf.plot(
        column=resid_col,
        ax=ax,
        cmap='RdBu_r',
        norm=TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs),
        linewidth=0.25,
        edgecolor='#4d4d4d',
    )
    ax.set_title(meta['label'])
    ax.set_axis_off()

sm = mpl.cm.ScalarMappable(cmap='RdBu_r', norm=TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs))
sm.set_array([])
fig.subplots_adjust(left=0.02, right=0.86, top=0.84, bottom=0.04, wspace=0.08)
cax = fig.add_axes([0.89, 0.16, 0.025, 0.62])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('SLM residual')
fig.suptitle('Spatial lag residuals, Jurong Lake Gardens', y=0.96)
save_figure(fig, 'jurong_slm_residual_maps')
display(fig)
plt.close(fig)


In [ ]:

exported = sorted(p.name for p in OUTPUT_DIR.glob('jurong_slm_*'))
for name in exported:
    print(name)
